# 7) Bug Hunt Bingo Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Make debugging fun: generate a 3x3 Bingo card of common bugs and let students mark progress.

## Simple rules

- Create a new card if none exists.
- Toggle marks by typing 1-9.
- Save progress in memory.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "07_bingo_memory.json"

BUGS = [
    "Off-by-one", "Wrong base case", "Wrong loop condition",
    "Indentation error", "Variable shadowing", "Forgot return",
    "Mutating list while iterating", "Wrong index", "Type mismatch",
    "Uninitialized variable", "Wrong comparison", "Infinite loop",
]

def render(grid: List[str], marked: set) -> str:
    out = []
    for i, item in enumerate(grid, start=1):
        tag = "X" if i in marked else " "
        out.append(f"[{tag}] {i}. {item}")
    return "\n".join(out)

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}
    grid = memory.get("grid") or random.sample(BUGS, 9)
    marked = set(memory.get("marked", []))

    Tools.notify("Bug Hunt Bingo Agent is running.")
    Tools.notify("Type 1-9 to toggle mark. Type 'stop' to end.")
    Tools.notify("Card:\n" + render(grid, marked))

    while True:
        cmd = Tools.ask("Command (1-9 / stop):")
        if should_stop(cmd):
            memory["grid"] = grid
            memory["marked"] = sorted(marked)
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved. Bye!")
            break
        if cmd.isdigit():
            n = int(cmd)
            if 1 <= n <= 9:
                if n in marked: marked.remove(n)
                else: marked.add(n)
                Tools.notify("Updated:\n" + render(grid, marked))
                memory["grid"] = grid
                memory["marked"] = sorted(marked)
                env.execute(Action("save_memory", {}), memory)

run_agent()


## Easy extensions

- Detect Bingo (row/col/diag) and celebrate.
- Let students customize bug list.
- Export the card to a text file.